#This fraud detection dataset contains transaction details to identify fraudulent activities. I handled missing values, converted categorical variables, and extracted useful features from date-time and age. To prevent errors, I standardized numerical data, reduced high-cardinality categories, and optimized memory usage. Finally, I trained a Logistic Regression model for fraud prediction

In [ ]:
import pandas as pd  # For handling data
import numpy as np   # For numerical operations
import seaborn as sns  # For visualization
import matplotlib.pyplot as plt  # For plotting graphs
from sklearn.model_selection import train_test_split  # To split data
from sklearn.preprocessing import StandardScaler  # For scaling features
from sklearn.linear_model import LogisticRegression  # The model
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report  # To evaluate the model
import joblib #For saving the model


Loading the dataset

In [ ]:
file_path = "/content/fraudTrain.csv"
df = pd.read_csv(file_path)

print(df.head())

   Unnamed: 0 trans_date_trans_time            cc_num  \
0           0   2019-01-01 00:00:18  2703186189652095   
1           1   2019-01-01 00:00:44      630423337322   
2           2   2019-01-01 00:00:51    38859492057661   
3           3   2019-01-01 00:01:16  3534093764340240   
4           4   2019-01-01 00:03:06   375534208663984   

                             merchant       category     amt      first  \
0          fraud_Rippin, Kub and Mann       misc_net    4.97   Jennifer   
1     fraud_Heller, Gutmann and Zieme    grocery_pos  107.23  Stephanie   
2                fraud_Lind-Buckridge  entertainment  220.11     Edward   
3  fraud_Kutch, Hermiston and Farrell  gas_transport   45.00     Jeremy   
4                 fraud_Keeling-Crist       misc_pos   41.96      Tyler   

      last gender                        street  ...      lat      long  \
0    Banks      F                561 Perry Cove  ...  36.0788  -81.1781   
1     Gill      F  43039 Riley Greens Suite 393  ...  48

Data Preprocessing to remove unneccessary columns

In [ ]:
df = df.drop(columns=['trans_num', 'cc_num', 'first', 'last', 'street', 'city', 'state', 'zip', 'job', 'dob'])

# Reducing the number of unique merchants
top_merchants = df['merchant'].value_counts().nlargest(50).index  # Keeping only top 50 merchants
df['merchant'] = df['merchant'].apply(lambda x: x if x in top_merchants else 'other')

# Converting 'trans_date_trans_time' to separate numerical features
df['transaction_year'] = pd.to_datetime(df['trans_date_trans_time']).dt.year
df['transaction_month'] = pd.to_datetime(df['trans_date_trans_time']).dt.month
df['transaction_day'] = pd.to_datetime(df['trans_date_trans_time']).dt.day
df['transaction_hour'] = pd.to_datetime(df['trans_date_trans_time']).dt.hour

# Dropping the original datetime column
df = df.drop(columns=['trans_date_trans_time'])


# Reducing the number of unique categories
top_categories = df['category'].value_counts().nlargest(20).index  # Keep only top 20 categories
df['category'] = df['category'].apply(lambda x: x if x in top_categories else 'other')

# Converting categorical variables to numerical values for more simpification
df = pd.get_dummies(df, columns=['category', 'merchant', 'gender'], drop_first=True)

Standardize (scale) the numerical features for better model performance

In [ ]:
# Standardizing only the numerical columns
numerical_cols = ['amt', 'lat', 'long']
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])


Defining the x and y variable for clarification

In [ ]:
X = df.drop(columns=['is_fraud'])  # Independent variable
y = df['is_fraud']  # Target variable (1 = Fraud, 0 = Not Fraud)

Splitting the data into training(80%) and testing(20%) sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

Initializing and training the Logistic Regression model

In [ ]:
model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)

LogisticRegression(max_iter=500)

Making the predictions

In [ ]:
y_pred = model.predict(X_test)

Cheching the accuracy of the model

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {(accuracy)*100}%")

Model Accuracy: 99.24559774058419%


Saving the trained model and scaler using joblib to access it later on when needed

In [ ]:
import joblib

joblib.dump(model, 'Credit_Card_Fraud.pkl')

# Save the fitted scaler
joblib.dump(scaler, 'scaler.pkl')

['scaler.pkl']

Creating a function to predict whether a given transaction is fraudulent or not.

Parameters:
model - Trained Logistic Regression model
scaler - Fitted StandardScaler for numerical features
transaction - Dictionary containing transaction details
feature_order - List of features used during model training
Will return :- "Fraud" or "Not Fraud"

In [ ]:
def predict_fraud(model, scaler, transaction, feature_order):

    # Converting input into a DataFrame
    input_df = pd.DataFrame([transaction])

    # Extracting and processing datetime features
    input_df['transaction_year'] = pd.to_datetime(input_df['trans_date_trans_time']).dt.year
    input_df['transaction_month'] = pd.to_datetime(input_df['trans_date_trans_time']).dt.month
    input_df['transaction_day'] = pd.to_datetime(input_df['trans_date_trans_time']).dt.day
    input_df['transaction_hour'] = pd.to_datetime(input_df['trans_date_trans_time']).dt.hour
    input_df = input_df.drop(columns=['trans_date_trans_time'])  # Drop datetime column

    # Applying categorical encoding
    input_df['merchant'] = input_df['merchant'].apply(lambda x: x if x in top_merchants else 'other')
    input_df['category'] = input_df['category'].apply(lambda x: x if x in top_categories else 'other')
    input_df = pd.get_dummies(input_df, columns=['category', 'merchant', 'gender'], drop_first=True)

    # Adding missing columns with default values (0)
    for col in feature_order:
        if col not in input_df.columns:
            input_df[col] = 0

    # Ensuring the order of features matches the training model
    input_df = input_df[feature_order]

    # Standardizing the numerical features
    numerical_cols = ['amt', 'lat', 'long']  # No 'age' since DOB is removed
    input_df[numerical_cols] = scaler.transform(input_df[numerical_cols])

    # Making the prediction
    prediction = model.predict(input_df)

    return "Fraud" if prediction[0] == 1 else "Not Fraud"



Using an example to see if the model works fine or not

In [ ]:
# Get feature order from training data
feature_order = X_train.columns.tolist()

# Example transaction (without DOB)
test_transaction = {
    "trans_date_trans_time": "2025-01-01 12:30:00",
    "category": "shopping_net",
    "amt": 1000.50,
    "merchant": "Amazon",
    "gender": "M",
    "lat": 40.7128,
    "long": -74.0060
}

# Predict fraud status
result = predict_fraud(model, scaler, test_transaction, feature_order)
print("Transaction Status:", result)


Transaction Status: Fraud
